In [3]:
import langgraph
import importlib.metadata

try:
    version = importlib.metadata.version('langgraph')
    print(f"This tutorial uses LangGraph version: {version}")
except importlib.metadata.PackageNotFoundError:
    print("LangGraph is installed but version info not available")

This tutorial uses LangGraph version: 1.1.6


In [4]:
pip install langgraph langchain-openai python-dotenv


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
print("All imports successful")


C:\Users\SAURABH\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
C:\Users\SAURABH\AppData\Roaming\Python\Python314\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


All imports successful


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print(f"API key loaded: {'Yes' if api_key else 'No -- check your .env file'}")


API key loaded: No -- check your .env file


In [8]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
import operator

class State(TypedDict):
    messages: Annotated[list, operator.add]


In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class GreetingState(TypedDict):
    name: str
    greeting: str


def format_name(state: GreetingState) -> dict:
    """Capitalize the user's name."""
    return {"name": state["name"].strip().title()}


def generate_greeting(state: GreetingState) -> dict:
    """Create a greeting using the formatted name."""
    return {"greeting": f"Hello, {state['name']}! Welcome to LangGraph."}


In [12]:
graph = StateGraph(GreetingState)

graph.add_node("format_name", format_name)
graph.add_node("generate_greeting", generate_greeting)

graph.add_edge(START, "format_name")
graph.add_edge("format_name", "generate_greeting")
graph.add_edge("generate_greeting", END)

print("Graph built with 2 nodes and 3 edges")


Graph built with 2 nodes and 3 edges


In [14]:
app = graph.compile()

result = app.invoke({"name": "  alice  ", "greeting": ""})
print(result)


{'name': 'Alice', 'greeting': 'Hello, Alice! Welcome to LangGraph.'}


In [18]:
# Initialize the model pointing to OpenRouter
llm = ChatOpenAI(
    model="google/gemini-2.5-pro",                     # Any model available on OpenRouter
    api_key="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",            # Your OpenRouter API key
    base_url="https://openrouter.ai/api/v1",           # OpenRouter target endpoint
    temperature=0.7
)

In [17]:
#Build a Graph That Calls an LLM

import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class ChatState(TypedDict):
    messages: Annotated[list, operator.add]

def call_llm(state: ChatState) -> dict:
    """Send messages to the LLM and append the response."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


In [19]:
chat_graph = StateGraph(ChatState)
chat_graph.add_node("call_llm", call_llm)
chat_graph.add_edge(START, "call_llm")
chat_graph.add_edge("call_llm", END)

chat_app = chat_graph.compile()


In [20]:
result = chat_app.invoke({
    "messages": [HumanMessage(content="What is LangGraph in one sentence?")]
})

print(result["messages"][-1].content)


LangGraph is a library for building stateful, agent-like applications by representing them as a graph, which allows for cycles to create more complex and controllable logic than a standard chain.


In [22]:
pip install grandalf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
print(chat_app.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| call_llm |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [24]:
print(chat_app.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	call_llm(call_llm)
	__end__([<p>__end__</p>]):::last
	__start__ --> call_llm;
	call_llm --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [29]:

# Display the graph as Mermaid diagram
from IPython.display import Markdown
mermaid_diagram = """```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
graph TD;
    __start__([<p>__start__</p>]):::first
    call_llm(call_llm)
    __end__([<p>__end__</p>]):::last
    __start__ --> call_llm;
    call_llm --> __end__;
    classDef default fill:#f2f0ff,line-height:1.2
    classDef first fill:#DAE8FC
    classDef last fill:#baffc9
```"""
display(Markdown(mermaid_diagram))

```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
graph TD;
    __start__([<p>__start__</p>]):::first
    call_llm(call_llm)
    __end__([<p>__end__</p>]):::last
    __start__ --> call_llm;
    call_llm --> __end__;
    classDef default fill:#f2f0ff,line-height:1.2
    classDef first fill:#DAE8FC
    classDef last fill:#baffc9
```

In [30]:
debug_app = chat_graph.compile(debug=True)


In [31]:
def call_llm_debug(state: ChatState) -> dict:
    """Send messages to LLM with debug printing."""
    print(f"[DEBUG] Input messages: {len(state['messages'])}")
    response = llm.invoke(state["messages"])
    print(f"[DEBUG] Response type: {type(response)}")
    return {"messages": [response]}


In [32]:
# Anthropic
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-4-20250514")

# Local via Ollama
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3")


ModuleNotFoundError: No module named 'langchain_ollama'

In [33]:
# Recommended approach
class State(TypedDict):
    messages: Annotated[list, operator.add]

graph = StateGraph(State)
